# Lesson 24: Image Stitching and Mosaicking

This lesson is a capstone for the projective-geometry block: it assembles feature detection and matching (Lesson 19), homography estimation (Lesson 22), and robust fitting (Lesson 23) into a complete pipeline that stitches two overlapping photos into a single seamless panorama.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## The pipeline, at a glance

1. **Detect and match features** between the two photos (Lesson 19: SIFT + ratio test).
2. **Estimate a homography** relating one image's plane to the other's, robustly (Lesson 22's homography, Lesson 23's RANSAC).
3. **Warp** one image into the other's coordinate frame (Lesson 9: `cv2.warpPerspective`).
4. **Composite and blend** the two images onto one canvas, feathering across the overlap so the seam is invisible.

## A synthetic panning camera

We build one wide synthetic scene, then take two overlapping horizontal crops from it &mdash; simulating a camera panning sideways between two shots. Since we generated the whole scene, we always have ground truth to check the stitched result against.

In [ ]:
rng = np.random.default_rng(0)
world = np.zeros((300, 600, 3), dtype=np.uint8)
for _ in range(40):
    x, y = rng.integers(20, 580), rng.integers(20, 280)
    radius = rng.integers(6, 20)
    color = tuple(int(v) for v in rng.integers(80, 255, 3))
    cv2.circle(world, (x, y), radius, color, -1)
for _ in range(20):
    x1, y1 = rng.integers(0, 600), rng.integers(0, 300)
    x2, y2 = rng.integers(0, 600), rng.integers(0, 300)
    color = tuple(int(v) for v in rng.integers(80, 255, 3))
    cv2.line(world, (x1, y1), (x2, y2), color, 2)

view1 = world[:, 0:400].copy()   # "left" photo
view2 = world[:, 200:600].copy() # "right" photo -- overlaps view1 by 200 columns

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(view1, cv2.COLOR_BGR2RGB))
axes[0].set_title('View 1')
axes[1].imshow(cv2.cvtColor(view2, cv2.COLOR_BGR2RGB))
axes[1].set_title('View 2')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 1: feature matching

In [ ]:
gray1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

bf = cv2.BFMatcher()
raw_matches = bf.knnMatch(des1, des2, k=2)
good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]

print(f'keypoints: {len(kp1)} (view 1), {len(kp2)} (view 2)')
print(f'good matches after ratio test: {len(good_matches)}')

match_vis = cv2.drawMatches(view1, kp1, view2, kp2, good_matches[:60], None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(11, 4))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title('Feature matches in the overlap region (first 60 shown)')
plt.axis('off')
plt.show()

## Step 2: robust homography

We solve for the homography that maps points in view 2 into view 1's coordinate frame, so warping view 2 through it lands it in the right place on a shared canvas.

In [ ]:
pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

H, inlier_mask = cv2.findHomography(pts2, pts1, cv2.RANSAC, 3.0)

print(f'inliers: {int(inlier_mask.sum())} / {len(inlier_mask)}')
print('recovered homography:')
print(np.round(H, 4))
print()
print('Since view 2 is exactly view 1 shifted left by 200 pixels in the source scene,')
print('this should be very close to a pure translation by (+200, 0).')

## Steps 3-4: warp, composite, and blend

We warp view 2 onto a canvas large enough to hold both images, then blend the overlap region with a simple **feather**: a linear alpha ramp from "fully view 1" to "fully view 2" across the overlap, the same weighted-sum blending as `cv2.addWeighted` in Lesson 2, just with a spatially-varying weight instead of a constant one.

In [ ]:
canvas_w, canvas_h = 600, 300

canvas1 = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
canvas1[:, :400] = view1
warped2 = cv2.warpPerspective(view2, H, (canvas_w, canvas_h))

has1 = canvas1.sum(axis=2) > 0
has2 = warped2.sum(axis=2) > 0
overlap = has1 & has2

overlap_cols = np.where(overlap.any(axis=0))[0]
x_start, x_end = overlap_cols.min(), overlap_cols.max()
ramp = np.clip((np.arange(canvas_w) - x_start) / (x_end - x_start + 1e-6), 0, 1)

alpha = np.zeros((canvas_h, canvas_w), dtype=np.float32)
alpha[overlap] = np.broadcast_to(ramp, (canvas_h, canvas_w))[overlap]

stitched = canvas1.astype(np.float32) * (1 - alpha[..., None]) + warped2.astype(np.float32) * alpha[..., None]
stitched[has1 & ~has2] = canvas1[has1 & ~has2]   # regions covered only by view 1
stitched[has2 & ~has1] = warped2[has2 & ~has1]   # regions covered only by view 2
stitched = stitched.astype(np.uint8)

plt.figure(figsize=(10, 4))
plt.imshow(cv2.cvtColor(stitched, cv2.COLOR_BGR2RGB))
plt.title('Stitched panorama')
plt.axis('off')
plt.show()

### How close is it to the real scene?

Because we generated `world` ourselves, we can compare the stitched result directly against ground truth &mdash; something impossible with a real photograph.

In [ ]:
difference = np.abs(stitched.astype(int) - world.astype(int))
print(f'mean absolute pixel error vs. ground truth: {difference.mean():.2f}')
print(f'max absolute pixel error vs. ground truth:  {difference.max()}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(cv2.cvtColor(world, cv2.COLOR_BGR2RGB))
axes[0].set_title('Ground truth scene')
axes[1].imshow(difference.astype(np.uint8) * 3)
axes[1].set_title('Difference (x3, for visibility)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

The stitched panorama matches the true scene almost exactly; the only visible error is a faint outline around a few small shapes right at the seam, from sub-pixel interpolation during the warp &mdash; not from any error in the recovered geometry.

## In practice: `cv2.Stitcher`

OpenCV bundles this entire pipeline (plus more robust blending, exposure compensation, and support for many images at once, arranged in any configuration) behind a single high-level call.

In [ ]:
stitcher = cv2.Stitcher_create()
status, panorama = stitcher.stitch([view1, view2])

print('status:', 'OK' if status == cv2.Stitcher_OK else f'failed ({status})')
if status == cv2.Stitcher_OK:
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    plt.title('cv2.Stitcher result')
    plt.axis('off')
    plt.show()

### Exercise

1. Reduce the overlap between `view1` and `view2` (e.g. crop `view2 = world[:, 350:600]`, leaving only 50 columns of overlap). At what point does SIFT matching find too few good matches for `findHomography` to produce a reliable result?
2. Replace the linear feather with a hard cutoff (no blending: just pick whichever image covers each pixel, splitting the overlap down the middle) and compare the visible seam quality to the feathered version.
3. Introduce a small rotation between the two views (e.g. warp `view2` slightly with `cv2.warpAffine` before feeding it into the pipeline) instead of a pure pan. Does the recovered homography still stitch the images together correctly, and how does `H` change compared to the pure-translation case?